# Kubernetes Bootcamp for Python Developers

**30 lessons · Markdown explanations + commented examples · Flask + Redis practice project**

This notebook introduces Kubernetes from first principles and moves toward real deployment workflows. It is suitable after learning basic Docker concepts.

## Learning outcomes
- Understand clusters, nodes, Pods, Deployments, Services, ConfigMaps, Secrets, and storage.
- Use `kubectl` to inspect, deploy, scale, update, and troubleshoot workloads.
- Understand probes, resource requests/limits, namespaces, RBAC, HPA, quotas, and NetworkPolicy.
- Read and adapt Kubernetes YAML manifests.
- Deploy a small Flask + Redis application to a local cluster.

## Prerequisites
- Basic Docker: images, containers, ports, volumes, and Dockerfiles.
- Docker Desktop with Kubernetes, Minikube, or kind.
- `kubectl`; Helm is optional for the Helm lesson.

**Important:** The notebook displays commands and YAML examples; it does not automatically change your cluster. Review each command and confirm your current Kubernetes context before applying or deleting resources.

The companion folder `kubernetes_practice_project/` contains an actual Flask + Redis example and manifests. Follow its README to build the app image, load it into your chosen local cluster, deploy resources, and test with port-forwarding.

Do not use the sample manifests as a complete production configuration. Production clusters need environment-specific security, secrets management, backup, observability, networking, and deployment policies.


## Lesson 1: 1. What Kubernetes solves

Kubernetes (often abbreviated **K8s**) orchestrates containerized applications across a cluster of machines. It helps maintain the desired number of replicas, restart failed containers, provide service discovery, coordinate rollouts, and place workloads on available nodes.

Docker focuses on building and running containers; Kubernetes manages containerized workloads and their supporting resources across a cluster. Kubernetes is not required for every small application: Compose is often enough for a single-machine development stack.

In [ ]:
concepts = {  # Summarize major Kubernetes terms.
    "Cluster": "A Kubernetes environment containing a control plane and worker nodes.",  # The overall environment.
    "Node": "A machine that runs Kubernetes workloads.",  # A VM or physical/virtual server.
    "Pod": "The smallest deployable unit; it contains one or more closely related containers.",  # Shared network and storage context.
    "Deployment": "A controller that manages stateless Pod replicas and rollouts.",  # Desired replica count and updates.
    "Service": "A stable network endpoint for a set of Pods.",  # Discoverable access to changing Pods.
    "ConfigMap": "A resource for non-secret configuration.",  # Configuration separated from images.
}  # Finish the concept map.
for name, meaning in concepts.items():  # Iterate over concepts.
    print(f"{name}: {meaning}")  # Print each definition.

## Lesson 2: 2. Kubernetes architecture

The **control plane** makes cluster-level decisions and stores cluster state. Core parts include the API server, scheduler, controller manager, and a backing key-value store (commonly etcd). Worker nodes run workloads and include components such as kubelet and a container runtime; cluster networking enables Pod and Service communication.

You usually interact with the API server using `kubectl`. Controllers continually compare actual state with desired state and try to reconcile differences. This declarative model is central to Kubernetes: describe the desired state, then observe whether the cluster reaches it.

In [ ]:
architecture_roles = {  # Associate a component with its high-level responsibility.
    "API server": "Validates and exposes the Kubernetes API.",  # Main API entry point.
    "etcd": "Stores cluster state and configuration data.",  # Backing datastore.
    "Scheduler": "Chooses a suitable node for an unscheduled Pod.",  # Pod placement.
    "Controller manager": "Runs controllers that reconcile desired and observed state.",  # Continuous reconciliation.
    "kubelet": "Ensures assigned Pods are running on a node.",  # Node agent.
    "Container runtime": "Runs containers, commonly through CRI-compatible software.",  # Container execution.
}  # Finish the architecture dictionary.
for component, role in architecture_roles.items():  # Iterate over components.
    print(f"{component}: {role}")  # Display each responsibility.

## Lesson 3: 3. Check kubectl and current context

`kubectl` is the command-line client for the Kubernetes API. `kubectl version --client` checks the client binary. `kubectl config get-contexts` lists saved contexts, and `kubectl config current-context` shows which context commands will target.

Always check the current context before applying or deleting resources. A context can point to a local practice cluster, a test environment, or production. The notebook shows commands as text; it does not run them against your cluster.

In [ ]:
kubectl_commands = [  # A safe initial diagnostic sequence.
    "kubectl version --client",  # Check the client version.
    "kubectl config get-contexts",  # List configured clusters and contexts.
    "kubectl config current-context",  # Identify the active context.
    "kubectl cluster-info",  # Check API endpoint and core control-plane information.
    "kubectl get nodes",  # Check whether nodes are visible and ready.
]  # Finish the command sequence.
print("\n".join(kubectl_commands))  # Display commands to run in a terminal.
print("Before applying manifests, confirm the active context is your intended learning cluster.")  # Highlight context safety.

## Lesson 4: 4. Core Kubernetes objects

Kubernetes uses API objects to represent desired state. A manifest commonly contains `apiVersion`, `kind`, `metadata`, and `spec`. The `spec` describes the desired configuration; the cluster's controllers work toward that state, while `status` reports what is happening.

Most resource types have clear responsibilities: Deployments manage stateless replicas; Services give Pods stable access; ConfigMaps and Secrets provide configuration; PVCs request persistent storage; Jobs perform finite work; CronJobs schedule repeated jobs.

In [ ]:
manifest_outline = {  # Show the typical Kubernetes manifest structure.
    "apiVersion": "apps/v1",  # Select the API group and version.
    "kind": "Deployment",  # Choose the object type.
    "metadata": {"name": "sample-web", "namespace": "default"},  # Identify the object.
    "spec": {"replicas": 2, "selector": {"matchLabels": {"app": "sample-web"}}},  # Sketch the desired state.
}  # Finish the illustrative object.
for key, value in manifest_outline.items():  # Iterate through manifest fields.
    print(f"{key}: {value}")  # Display each field.
print("A real Deployment also needs a Pod template whose labels match the selector.")  # Highlight a required relationship.

## Lesson 5: 5. Pods and containers

A Pod is the smallest deployable unit in Kubernetes. It may hold one container or a small set of tightly coupled containers that share a network namespace and can share volumes. A Pod gets an IP address, but that IP is not usually treated as a permanent application endpoint because Pods are replaceable.

Avoid using standalone Pods for workloads that need automatic replacement. A Deployment or another controller should normally manage Pods. When a Pod fails, the controller's desired-state behavior can create a replacement.

In [ ]:
pod_yaml = """apiVersion: v1
kind: Pod
metadata:
  name: hello-pod
  labels:
    app: hello
spec:
  containers:
    - name: web
      image: nginx:stable
      ports:
        - containerPort: 80
"""  # Define a minimal example Pod.
print(pod_yaml)  # Display the YAML.
pod_commands = ["kubectl apply -f pod.yaml", "kubectl get pods -o wide", "kubectl describe pod hello-pod", "kubectl delete pod hello-pod"]  # List commands for inspection.
print("\n".join(pod_commands))  # Display commands; they are not executed automatically.

## Lesson 6: 6. Labels and selectors

Labels are key-value metadata used to organize and select Kubernetes objects. A Service usually uses a label selector to identify the Pods that should receive traffic. Deployment selectors must match the labels on the Pod template.

A selector mismatch can lead to a Service with no endpoints or a Deployment that is invalid. Use labels consistently, for example `app: web` for application identity and `app.kubernetes.io/part-of` for broader grouping.

In [ ]:
labels = {"app": "web", "environment": "dev", "app.kubernetes.io/part-of": "sales-demo"}  # Define a set of descriptive labels.
selector = {"app": "web"}  # Define a selector expected by a Service or Deployment.
matches = all(labels.get(key) == value for key, value in selector.items())  # Check whether all selector pairs match.
print("Labels:", labels)  # Display the labels.
print("Selector:", selector)  # Display the selector.
print("Matches selector:", matches)  # Confirm whether the resource is selected.

## Lesson 7: 7. Deployments and ReplicaSets

A Deployment manages a desired number of replicas and provides declarative updates for stateless applications. It creates and manages ReplicaSets, which maintain the requested count of Pods for a particular revision.

Use a Deployment for a web API, frontend, or other stateless process. For a workload that requires stable per-instance identity or storage behavior, a StatefulSet may be more appropriate. Choose controllers based on application semantics rather than treating every workload as interchangeable.

In [ ]:
deployment_yaml = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 3
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: web
          image: nginx:stable
          ports:
            - containerPort: 80
"""  # Define a small three-replica Deployment.
print(deployment_yaml)  # Display the manifest.
print("Apply with: kubectl apply -f deployment.yaml")  # Show the command without applying it.

## Lesson 8: 8. Services: ClusterIP, NodePort, LoadBalancer

A Service provides a stable virtual endpoint for Pods selected by labels. `ClusterIP` is the common default for internal cluster access. `NodePort` exposes a port on cluster nodes, and `LoadBalancer` asks the environment's supported infrastructure to provide a load balancer.

The correct choice depends on the environment and exposure needs. A local development app can often be tested with `kubectl port-forward` without publishing a permanent external endpoint. Production exposure may use an Ingress controller or Gateway API implementation.

In [ ]:
service_examples = {  # Compare common Service types.
    "ClusterIP": "Internal cluster endpoint; typical for databases and internal APIs.",  # Internal traffic.
    "NodePort": "Exposes a port on each node; often used in labs or special environments.",  # Node-level port.
    "LoadBalancer": "Requests an external load balancer when the platform supports it.",  # Cloud or compatible environment.
}  # Finish the comparison.
for service_type, purpose in service_examples.items():  # Display the differences.
    print(f"{service_type}: {purpose}")  # Print the use case.
print("Local test command: kubectl port-forward service/web 8080:80")  # Show a temporary local access method.

## Lesson 9: 9. ConfigMaps

A ConfigMap stores non-sensitive configuration separately from container images. It can provide environment variables or mounted configuration files. The application must still read those settings correctly, and changes to an environment variable sourced from a ConfigMap generally require Pods to be restarted before they see the new value.

Do not store passwords, tokens, private keys, or other secrets in ConfigMaps. Use Kubernetes Secrets or a proper external secrets integration for sensitive values, with access controls and appropriate encryption configuration.

In [ ]:
configmap_yaml = """apiVersion: v1
kind: ConfigMap
metadata:
  name: web-config
data:
  APP_ENV: "development"
  REDIS_HOST: "redis"
  REDIS_PORT: "6379"
"""  # Define non-secret application settings.
print(configmap_yaml)  # Display the ConfigMap manifest.
print("Apply: kubectl apply -f configmap.yaml")  # Show the apply command.
print("Inspect: kubectl describe configmap web-config")  # Show how to verify its contents.

## Lesson 10: 10. Secrets and sensitive configuration

Kubernetes Secrets are intended for small amounts of sensitive configuration, such as passwords and tokens. Values represented with Base64 in a YAML manifest are encoded, not encrypted. Protect access with RBAC, configure encryption at rest where supported, and avoid committing real secret values to version control.

For actual deployment, prefer a secure secret injection workflow or an external secrets manager. The example uses a deliberately fake placeholder, and it should never be replaced with a real credential inside a public learning repository.

In [ ]:
secret_yaml = """apiVersion: v1
kind: Secret
metadata:
  name: example-secret
type: Opaque
stringData:
  username: demo-user
  password: REPLACE_WITH_A_LOCAL_TEST_VALUE
"""  # Use only a placeholder to demonstrate the resource shape.
print(secret_yaml)  # Display the example manifest; do not put production secrets here.
print("Restrict Secret access with RBAC and keep real secrets out of Git.")  # State the key security practices.

## Lesson 11: 11. Probes: liveness, readiness, startup

**Liveness probes** check whether a container should be restarted. **Readiness probes** determine whether it should receive normal traffic. **Startup probes** give slow-starting applications more time before liveness checks apply.

Choose paths and thresholds based on actual application behavior. A liveness probe that depends on a remote database can restart a healthy application during a database outage, creating a restart storm. Readiness checks can include dependencies when the application should not receive requests unless those dependencies are available.

In [ ]:
probe_yaml = """readinessProbe:
  httpGet:
    path: /ready
    port: http
  initialDelaySeconds: 5
  periodSeconds: 5
livenessProbe:
  httpGet:
    path: /health
    port: http
  initialDelaySeconds: 10
  periodSeconds: 10
"""  # Define example readiness and liveness checks.
print(probe_yaml)  # Display the YAML fields.
print("Readiness controls traffic eligibility; liveness can trigger a container restart.")  # Explain the distinction.

## Lesson 12: 12. CPU and memory requests and limits

Resource **requests** help the scheduler decide where a Pod can fit and are used in resource allocation decisions. **Limits** constrain resource use: a CPU limit can throttle a process, and exceeding a memory limit can cause the container to be killed with an out-of-memory event.

Set requests and limits based on observed workload behavior and operational requirements. Tiny limits can cause slow responses or restarts; excessive requests can reduce how many Pods fit on available nodes. Monitoring and load testing help tune values.

In [ ]:
resources_yaml = """resources:
  requests:
    cpu: "100m"
    memory: "128Mi"
  limits:
    cpu: "500m"
    memory: "512Mi"
"""  # Define example resource requests and limits.
print(resources_yaml)  # Display the resource configuration.
print("100m CPU is one tenth of a CPU core; memory units such as Mi are binary units.")  # Explain common units.

## Lesson 13: 13. Scale workloads

Manual scaling changes the desired number of replicas for a controller. Kubernetes then creates or removes Pods to approach that replica count. More replicas can improve availability or throughput, but only if the application is appropriately designed and the cluster has enough capacity.

Stateless application replicas are often easy to scale. Stateful components need extra care around data consistency, identity, and storage. Scaling a deployment does not automatically make a database or a shared dependency scale safely.

In [ ]:
scaling_commands = [  # Show common scaling and observation commands.
    "kubectl -n k8s-learning scale deployment/web --replicas=3",  # Set the desired number of web replicas.
    "kubectl -n k8s-learning get deployment web",  # Check desired and available replicas.
    "kubectl -n k8s-learning get pods -l app=web -o wide",  # List matching Pods and their nodes.
]  # Finish the command list.
print("\n".join(scaling_commands))  # Display commands for manual execution.

## Lesson 14: 14. Rolling updates and rollbacks

A Deployment can update an image or other Pod-template field using a rolling update strategy. Kubernetes replaces old Pods with new ones according to the deployment strategy and readiness status. If the new version fails readiness or behaves incorrectly, a rollback may be needed.

Use immutable release tags rather than reusing the same tag for different builds, and inspect rollout status. Kubernetes' rollout history is useful, but an application rollback may also require compatible database migrations, configuration changes, or other system-level actions.

In [ ]:
rollout_commands = [  # Common deployment update commands.
    "kubectl -n k8s-learning set image deployment/web web=k8s-flask-redis:1.1.0",  # Update the image to a new release tag.
    "kubectl -n k8s-learning rollout status deployment/web",  # Wait for rollout progress.
    "kubectl -n k8s-learning rollout history deployment/web",  # Inspect saved rollout revisions.
    "kubectl -n k8s-learning rollout undo deployment/web",  # Roll back to the previous revision if appropriate.
]  # Finish rollout commands.
print("\n".join(rollout_commands))  # Display commands.
print("The named image must exist in the cluster's image source; the example tag is illustrative.")  # Explain image availability.

## Lesson 15: 15. Logs, exec, describe, and events

Use `kubectl logs` for container output, `kubectl exec` to start a command in a running container, `kubectl describe` for object details and events, and `kubectl get events` for recent cluster events. For a Pod with multiple containers, specify the container name where needed.

For a crashing container, inspect its current and previous logs. A `CrashLoopBackOff` state describes repeated failed starts and backoff; it is a symptom, not the root cause. Check configuration, startup command, environment variables, mounts, permissions, and dependency availability.

In [ ]:
debug_commands = [  # Diagnostics for a workload in the learning namespace.
    "kubectl -n k8s-learning get pods",  # Check Pod status.
    "kubectl -n k8s-learning describe pod <pod-name>",  # Inspect conditions, events, and scheduling details.
    "kubectl -n k8s-learning logs <pod-name>",  # Read current logs.
    "kubectl -n k8s-learning logs <pod-name> --previous",  # Read logs from the previous crashed container instance.
    "kubectl -n k8s-learning exec -it <pod-name> -- sh",  # Open a shell in a running container if the image includes one.
    "kubectl -n k8s-learning get events --sort-by=.lastTimestamp",  # Review recent namespace events.
]  # Finish the diagnostics.
print("\n".join(debug_commands))  # Display the commands for manual use.

## Lesson 16: 16. Namespaces and context safety

Namespaces divide many Kubernetes resources into logical groups. They are useful for separating projects and environments, but they are not by themselves a complete security boundary. RBAC, network policies, resource quotas, and other controls may also be needed.

Use `-n` or `--namespace` to target a specific namespace. Before destructive operations, confirm the current context and the resource scope. Avoid `kubectl delete all --all` or broad commands unless you fully understand what they affect; the resource alias `all` does not include every Kubernetes resource type.

In [ ]:
namespace_commands = [  # Namespace and context inspection commands.
    "kubectl config current-context",  # Confirm the cluster target.
    "kubectl get namespaces",  # List namespaces.
    "kubectl -n k8s-learning get all",  # List common workload resources in the project namespace.
    "kubectl -n k8s-learning get configmaps,secrets,pvc",  # Inspect additional configuration and storage resources.
]  # Finish the list.
print("\n".join(namespace_commands))  # Display commands.
print("The namespace is not a substitute for authorization or network isolation.")  # Explain the limitation.

## Lesson 17: 17. PersistentVolumes and PersistentVolumeClaims

A PersistentVolume (PV) represents storage available to the cluster, while a PersistentVolumeClaim (PVC) requests storage with a size and access mode. A StorageClass can dynamically provision a PV when a matching claim is created.

A PVC can remain Pending if no suitable storage provisioner or volume exists. Access modes do not guarantee the same behavior across every storage system, and a volume's reclaim policy affects what happens after a claim is released. Verify cluster storage behavior before relying on persistence.

In [ ]:
pvc_yaml = """apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: app-data
spec:
  accessModes:
    - ReadWriteOnce
  resources:
    requests:
      storage: 1Gi
"""  # Define a request for one GiB of persistent storage.
print(pvc_yaml)  # Display the PVC manifest.
print("Inspect claims: kubectl get pvc")  # Show how to check claim status.
print("A default StorageClass is often needed for dynamic provisioning in local clusters.")  # Explain a common prerequisite.

## Lesson 18: 18. StatefulSet versus Deployment

Use a Deployment for interchangeable stateless replicas. A StatefulSet is designed for workloads that need stable network identities and/or stable per-replica storage associations. StatefulSets commonly use headless Services and `volumeClaimTemplates`.

A StatefulSet does not automatically make an application distributed or highly available. Databases may require replication, leader election, backup, recovery, and application-specific operational procedures in addition to the Kubernetes object.

In [ ]:
controller_guide = {  # Choose a controller by workload behavior.
    "Deployment": "Stateless web API, frontend, worker whose replicas are interchangeable.",  # Replicas can be replaced independently.
    "StatefulSet": "Workload needing stable Pod identity or per-replica storage.",  # Stateful identity and storage association.
    "Job": "A task that should run to completion.",  # Finite batch work.
    "CronJob": "A task scheduled to run periodically.",  # Recurring batch work.
    "DaemonSet": "A Pod that should run on selected nodes.",  # Node-level agent.
}  # Finish the guide.
for controller, use_case in controller_guide.items():  # Display each controller.
    print(f"{controller}: {use_case}")  # Show a typical use case.

## Lesson 19: 19. Jobs and CronJobs

A Job manages Pods that run a finite task to completion, such as a data import or report generation. A CronJob creates Jobs on a schedule. Configure time limits, retry behavior, parallelism, and history limits according to the task's risk and duration.

Scheduled jobs can overlap or run late depending on configuration and cluster availability. Design jobs to be idempotent where possible—re-running them should not corrupt data or duplicate effects.

In [ ]:
cronjob_yaml = """apiVersion: batch/v1
kind: CronJob
metadata:
  name: report-summary
spec:
  schedule: "0 6 * * 1"
  concurrencyPolicy: Forbid
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 2
  jobTemplate:
    spec:
      backoffLimit: 2
      template:
        spec:
          restartPolicy: Never
          containers:
            - name: report
              image: python:3.12-slim
              command: ["python", "-c", "print('generate report here')"]
"""  # Define a Monday 06:00 UTC scheduled demo.
print(cronjob_yaml)  # Display the CronJob manifest.
print("Confirm the cluster timezone convention; cron schedules are interpreted by the CronJob controller's configured timezone behavior.")  # Note scheduling context.

## Lesson 20: 20. Ingress and external HTTP access

An Ingress object defines HTTP(S) routing rules, but it does not provide an ingress controller on its own. The cluster must have a supported controller installed and configured. Kubernetes Gateway API is another option in environments that support it.

For a local practice service, `kubectl port-forward service/web 8080:80` is often simpler. For production, consider TLS, DNS, authentication, request limits, and the controller's security configuration. Do not expose internal services unnecessarily.

In [ ]:
ingress_yaml = """apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: web
spec:
  ingressClassName: nginx
  rules:
    - host: demo.example.test
      http:
        paths:
          - path: /
            pathType: Prefix
            backend:
              service:
                name: web
                port:
                  number: 80
"""  # Define an example ingress rule; it assumes a matching controller and DNS setup.
print(ingress_yaml)  # Display the YAML.
print("Local alternative: kubectl -n k8s-learning port-forward service/web 8080:80")  # Show simpler local access.

## Lesson 21: 21. Service DNS and discovery

Services provide stable DNS names within a cluster. A Pod can typically connect to a Service by its name in the same namespace. From another namespace, the fully qualified service name commonly includes `<service>.<namespace>.svc.cluster.local`, subject to cluster DNS configuration.

Do not hard-code Pod IP addresses: they can change as Pods are recreated. Use the Service name for internal dependencies. A DNS failure may be caused by a misspelled Service name, wrong namespace, missing endpoints, network policy, or cluster DNS problems.

In [ ]:
dns_examples = [  # Common name-resolution checks.
    "kubectl -n k8s-learning get service redis",  # Confirm the service exists.
    "kubectl -n k8s-learning get endpointslice -l kubernetes.io/service-name=redis",  # Check whether Service endpoints exist.
    "kubectl -n k8s-learning exec deploy/web -- python -c \"import socket; print(socket.gethostbyname('redis'))\"",  # Resolve the Service from a web container.
]  # Finish the commands.
print("\n".join(dns_examples))  # Display commands for manual troubleshooting.

## Lesson 22: 22. RBAC and ServiceAccounts

Role-Based Access Control (RBAC) defines who can perform which API actions on which resources. A ServiceAccount is an identity for a workload. Namespaced Roles and RoleBindings are appropriate when permissions should be limited to one namespace; ClusterRoles and ClusterRoleBindings grant broader scope.

Follow least privilege: give a workload only the API permissions it needs. Many applications do not need Kubernetes API access at all. Avoid assigning cluster-admin for convenience, especially to application ServiceAccounts.

In [ ]:
rbac_yaml = """apiVersion: v1
kind: ServiceAccount
metadata:
  name: report-reader
  namespace: k8s-learning
---
apiVersion: rbac.authorization.k8s.io/v1
kind: Role
metadata:
  name: pod-reader
  namespace: k8s-learning
rules:
  - apiGroups: [""]
    resources: ["pods"]
    verbs: ["get", "list", "watch"]
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: report-reader-pod-reader
  namespace: k8s-learning
subjects:
  - kind: ServiceAccount
    name: report-reader
    namespace: k8s-learning
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: Role
  name: pod-reader
"""  # Define minimal namespaced read access to Pods.
print(rbac_yaml)  # Display the RBAC example.
print("Do not grant write access or cluster-wide permissions unless required.")  # Emphasize least privilege.

## Lesson 23: 23. Autoscaling and HPA

A HorizontalPodAutoscaler adjusts a workload's replica count based on observed metrics, often CPU or memory utilization. Resource requests are important because utilization percentages are commonly calculated relative to those requests. The cluster also needs a working metrics API, such as metrics-server, for resource-based scaling.

Autoscaling has a response delay and cannot solve every bottleneck. If an application is limited by a database, an external API, or a shared lock, adding replicas may worsen contention. Configure min/max replicas and observe both app performance and dependency load.

In [ ]:
hpa_yaml = """apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: web
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: web
  minReplicas: 2
  maxReplicas: 5
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70
"""  # Define a CPU-based scaling target.
print(hpa_yaml)  # Display the HPA manifest.
print("Inspect status with: kubectl get hpa")  # Show how to check scaling activity.
print("This requires resource requests and a working metrics API.")  # Note the prerequisite.

## Lesson 24: 24. ResourceQuota and LimitRange

A ResourceQuota limits aggregate resource consumption or the number of certain objects in a namespace. A LimitRange can provide default or maximum/minimum resource requirements for individual Pods or containers. These tools help prevent one namespace or workload from using all available resources.

Quotas can block creation if a manifest requests resources beyond the remaining allowance. Defaults may be applied by admission controls, so inspect the effective Pod specification when debugging scheduling or quota errors.

In [ ]:
quota_yaml = """apiVersion: v1
kind: ResourceQuota
metadata:
  name: learning-quota
  namespace: k8s-learning
spec:
  hard:
    requests.cpu: "2"
    requests.memory: 2Gi
    limits.cpu: "4"
    limits.memory: 4Gi
    pods: "20"
"""  # Set illustrative namespace resource caps.
print(quota_yaml)  # Display the quota.
print("Check usage with: kubectl -n k8s-learning describe resourcequota learning-quota")  # Provide the inspection command.

## Lesson 25: 25. NetworkPolicy basics

NetworkPolicy can restrict traffic to and from selected Pods, but enforcement depends on the cluster network plugin. A policy that selects Pods without permitting necessary ingress or egress may break DNS, database connections, or application traffic.

Start with a clear traffic map: which Pods need to talk to which Services, on which ports, and which external endpoints are needed? Test policy changes in a learning namespace and have a rollback plan. NetworkPolicy is not effective protection if the cluster's CNI does not enforce it.

In [ ]:
network_policy_yaml = """apiVersion: networking.k8s.io/v1
kind: NetworkPolicy
metadata:
  name: web-ingress-from-namespace
  namespace: k8s-learning
spec:
  podSelector:
    matchLabels:
      app: web
  policyTypes:
    - Ingress
  ingress:
    - from:
        - namespaceSelector:
            matchLabels:
              kubernetes.io/metadata.name: k8s-learning
      ports:
        - protocol: TCP
          port: 5000
"""  # Restrict ingress to web Pods from the learning namespace on port 5000.
print(network_policy_yaml)  # Display the example policy.
print("This policy is only an example; test CNI support and preserve required health/traffic paths.")  # Add implementation caution.

## Lesson 26: 26. Helm and chart concepts

Helm packages Kubernetes resources into charts with templates and configurable values. A chart can be installed as a release, upgraded with new values or versions, and rolled back. Helm is useful when applications have many related manifests or need reusable configuration across environments.

Helm templates add a layer of abstraction: inspect rendered manifests before applying them. Don't adopt Helm merely for a single tiny manifest; plain YAML and Kustomize can be simpler for small projects.

In [ ]:
helm_commands = [  # Common Helm workflow commands.
    "helm repo add bitnami https://charts.bitnami.com/bitnami",  # Add a chart repository; review trust and chart details first.
    "helm repo update",  # Refresh repository metadata.
    "helm search repo redis",  # Search chart names available in the repository.
    "helm show values <repo>/<chart>",  # Inspect configurable values.
    "helm template demo <repo>/<chart> -f values.yaml",  # Render resources locally before installing.
    "helm list -n <namespace>",  # List releases in a namespace.
]  # Finish the workflow.
print("\n".join(helm_commands))  # Display commands for manual review.
print("Review the rendered YAML, chart version, permissions, images, and values before installing.")  # Emphasize inspection.

## Lesson 27: 27. Troubleshoot Pending Pods and CrashLoopBackOff

A `Pending` Pod may be waiting for scheduling or volume binding. Common causes include insufficient resources, node selectors, taints and tolerations, or a PVC that cannot bind. `CrashLoopBackOff` means the container repeatedly exits and Kubernetes delays restarts; the root cause may be an application error, invalid configuration, missing dependency, permission problem, or failed startup.

Use `kubectl describe pod` to read events, `kubectl logs --previous` for earlier crash output, and `kubectl get pvc` for storage state. Diagnose the underlying error rather than repeatedly deleting Pods.

In [ ]:
troubleshooting_commands = [  # Match a symptom to a useful diagnostic.
    "kubectl -n k8s-learning get pods -o wide",  # See statuses and assigned nodes.
    "kubectl -n k8s-learning describe pod <pod-name>",  # Review events, scheduling, probes, and conditions.
    "kubectl -n k8s-learning logs <pod-name> --previous",  # Inspect the previous crashed container.
    "kubectl -n k8s-learning get pvc",  # Check whether storage claims are bound.
    "kubectl describe node <node-name>",  # Review allocatable capacity, pressure, and taints.
    "kubectl -n k8s-learning get events --sort-by=.lastTimestamp",  # Inspect recent events.
]  # Finish the diagnostics.
print("\n".join(troubleshooting_commands))  # Display the sequence.

## Lesson 28: 28. Local Kubernetes options

Common ways to practice locally include Kubernetes in Docker Desktop, Minikube, and kind (Kubernetes in Docker). Each option has its own way of making locally built images available to the cluster and may differ in load balancers, storage provisioners, and add-ons.

For kind, a local image can be loaded with `kind load docker-image ...`; for Minikube, use `minikube image load ...`. Docker Desktop's integrated cluster behavior depends on its version and settings. Check current cluster documentation if a manifest is correct but a local image cannot be found.

In [ ]:
local_cluster_commands = {  # Show common cluster setup and image-loading commands.
    "Docker Desktop": ["kubectl config current-context", "kubectl get nodes"],  # Check the selected integrated cluster.
    "Minikube": ["minikube start", "minikube image load k8s-flask-redis:dev"],  # Start cluster and load a local image.
    "kind": ["kind create cluster", "kind load docker-image k8s-flask-redis:dev"],  # Create a cluster and load the local image.
}  # Finish the command map.
for environment, commands in local_cluster_commands.items():  # Iterate through local cluster choices.
    print(f"\n{environment}:")  # Display the environment name.
    for command in commands:  # Display the environment's commands.
        print(" ", command)  # Print one command.
print("Choose one environment; do not create multiple clusters unnecessarily.")  # Suggest a clear setup.

## Lesson 29: 29. Practice project: Flask + Redis

The companion project demonstrates a multi-tier app: two web Pods, one Redis Deployment, a ConfigMap for non-sensitive settings, a Service for each component, and a PVC for Redis data. The web Deployment defines resource requests/limits and liveness/readiness probes. Its Service provides a stable cluster endpoint.

Build the local app image first and load it into your local cluster if necessary. Apply the Namespace before namespaced resources. Check the PVC and Pods become ready before testing the service.

In [ ]:
project_files = [  # List the companion project's important files.
    "Dockerfile",  # Builds the Flask + Redis web image.
    "app.py",  # Implements /, /health, /ready, and /visits.
    "requirements.txt",  # Defines Flask and redis-py dependencies.
    "k8s/00-namespace.yaml",  # Creates the project namespace.
    "k8s/01-configmap.yaml",  # Configures the web service.
    "k8s/02-redis-pvc.yaml",  # Requests persistent Redis storage.
    "k8s/03-redis.yaml",  # Creates Redis Deployment and Service.
    "k8s/04-web.yaml",  # Creates web Deployment and Service.
    "k8s/05-hpa-optional.yaml",  # Optional autoscaling example; needs metrics API.
]  # Finish the manifest inventory.
for filename in project_files:  # Iterate through the files.
    print("-", filename)  # Display each file.
print("Run commands from kubernetes_practice_project/; see its README for cluster-specific image loading.")  # Point to instructions.

## Lesson 30: 30. Deploy, verify, scale, and clean up

A useful deployment loop is: validate manifests, apply them, wait for rollouts, check resource health, test the Service, and inspect logs. Scale and update workloads deliberately, then verify readiness and application behavior. Treat cleanup as a separate decision because deleting a namespace can also delete PVCs and other resources.

For a local exercise, use `kubectl port-forward` to reach the internal Service. When finished, delete only the practice namespace after confirming it holds no data you need.

In [ ]:
workflow_commands = [  # Practice sequence for the companion app.
    "kubectl apply --dry-run=client -f k8s/00-namespace.yaml",  # Validate the namespace manifest locally.
    "kubectl apply -f k8s/00-namespace.yaml",  # Create the namespace.
    "kubectl apply -f k8s/01-configmap.yaml -f k8s/02-redis-pvc.yaml -f k8s/03-redis.yaml -f k8s/04-web.yaml",  # Apply the app dependencies and web resources.
    "kubectl -n k8s-learning rollout status deployment/redis",  # Wait for Redis rollout.
    "kubectl -n k8s-learning rollout status deployment/web",  # Wait for the web rollout.
    "kubectl -n k8s-learning port-forward service/web 8080:80",  # Forward local port 8080 to the web Service.
    "kubectl delete namespace k8s-learning",  # Remove the practice namespace only when finished and data is disposable.
]  # Finish the example workflow.
print("\n".join(workflow_commands))  # Display commands but do not execute them automatically.